# resultm

In [ ]:
///- --test static

open testing

In [ ]:
open rust
open rust_operators

## resultm

### from_option_error

In [ ]:
inl from_option_error error opt =
    match opt with
    | Some x => Ok x
    | None => Error error

### from_option

In [ ]:
inl from_option opt =
    opt |> from_option_error "resultm.from_option / Option does not have a value."

### flatten_option

In [ ]:
inl flatten_option forall t u. (x : option (result (option t) u)) : result (option t) u =
    match x with
    | Some (Error x) => Error x
    | Some (Ok (Some x)) => Ok (Some x)
    | _ => Ok None

### flatten

In [ ]:
inl flatten forall t u. (x : result (result t u) u) : result t u =
    match x with
    | Ok x => x
    | Error x => Error x

### from_try

In [ ]:
inl from_try forall t. (fn : () -> t) (ex_fn : (() -> exn) -> exn) : result t exn =
    inl ok x = x |> Ok : result t exn
    inl error (e : exn) = e |> Error : result t exn
    backend_switch {
        Lua = fun () =>
            $'local resultm_fn = !fn '
            $'local resultm_ok = !ok '
            $'local resultm_error = !error '
            $'local resultm_ex_fn = !ex_fn '
            $'local status, result = pcall(resultm_fn)'
            $'local r = nil'
            $'if status then'
            $'    r = resultm_ok(result)'
            $'else'
            $'    r = resultm_error(resultm_ex_fn(function() return result end))'
            $'end'
            $'r' : result t exn
        Fsharp = fun () =>
            join $'try !fn () |> !ok with ex -> (fun () -> ex) |> !ex_fn |> !error ' : result t exn
        Python = fun () =>
            $'fn = !fn '
            $'ok = !ok '
            $'error = !error '
            $'ex_fn = !ex_fn '
            $'try: x = ok(fn()) '
            $'except Exception as ex: x = error(ex_fn(lambda: ex))'
            $'x' : result t exn
        TypeScript = fun () =>
            $'(() => { try { return !ok(!fn()); } catch (ex) { return !error(!ex_fn(() => ex)); } })()' : result t exn
        Zig = fun () => fn () |> ok
    }

### get

In [ ]:
inl get forall t e. (source : result t e) : t =
    match source with
    | Ok x => x
    | Error x =>
        backend_switch {
            Gleam = fun () =>
                global "import gleam/string"
                $'"resultm.get / Error x: " <> string.inspect(!x)' : string
            Lua = fun () => $'"resultm.get / Error x: " .. tostring(!x)' : string
            Fsharp = fun () => $'$"resultm.get / Error x: {!x}"' : string
            Python = fun () => $'f"resultm.get / Error x: {!x}"' : string
            TypeScript = fun () => $'"resultm.get / Error x: " + String(!x)' : string
            Rust = fun () => $'Rc::<str>::from({ struct SpiralDebug<T>(T); trait SpiralDebugShow { fn show(&self) -> String; } impl<T: std::fmt::Debug> SpiralDebugShow for SpiralDebug<T> { fn show(&self) -> String { format\!("{:?}", self.0) } } trait SpiralTypeShow { fn show(&self) -> String; } impl<T> SpiralTypeShow for &SpiralDebug<T> { fn show(&self) -> String { String::from(std::any::type_name::<T>()) } } format\!("resultm.get / Error x: {}", (&SpiralDebug(&!x)).show()) })' : string
            Zig = fun () => $'(std.fmt.allocPrint(spiral_gpa, "resultm.get / Error x: {any}", .{ !x }) catch \@panic("out of memory"))' : string
        }
        |> failwith

### map

In [ ]:
inl map forall t e u. (fn : t -> u) (source : result t e) : result u e =
    match source with
    | Ok x => x |> fn |> Ok
    | Error x => Error x

### map_error

In [ ]:
inl map_error forall t e u. (fn : e -> u) (source : result t e) : result t u =
    match source with
    | Ok x => Ok x
    | Error x => x |> fn |> Error

### unwrap_err

In [ ]:
inl unwrap_err forall t u. (x : result t u) : u =
    match x with
    | Ok x =>
        backend_switch {
            Gleam = fun () =>
                global "import gleam/string"
                $'"resultm.unwrap_err / Ok x: " <> string.inspect(!x)' : string
            Lua = fun () => $'"resultm.unwrap_err / Ok x: " .. tostring(!x)' : string
            Fsharp = fun () => $'$"resultm.unwrap_err / Ok x: {!x}"' : string
            Python = fun () => $'f"resultm.unwrap_err / Ok x: {!x}"' : string
            TypeScript = fun () => $'"resultm.unwrap_err / Ok x: " + String(!x)' : string
            Rust = fun () => $'Rc::<str>::from({ struct SpiralDebug<T>(T); trait SpiralDebugShow { fn show(&self) -> String; } impl<T: std::fmt::Debug> SpiralDebugShow for SpiralDebug<T> { fn show(&self) -> String { format\!("{:?}", self.0) } } trait SpiralTypeShow { fn show(&self) -> String; } impl<T> SpiralTypeShow for &SpiralDebug<T> { fn show(&self) -> String { String::from(std::any::type_name::<T>()) } } format\!("resultm.unwrap_err / Ok x: {}", (&SpiralDebug(&!x)).show()) })' : string
            Zig = fun () => $'(std.fmt.allocPrint(spiral_gpa, "resultm.unwrap_err / Ok x: {any}", .{ !x }) catch \@panic("out of memory"))' : string
        }
        |> failwith
    | Error x => x

### ok

In [ ]:
inl ok forall t. (x : result t _) : option t =
    match x with
    | Ok x => Some x
    | Error _ => None

## fsharp

### result'

In [ ]:
inl result'_types () =
    backend_switch {
        Gleam = fun () => global "import gleam/result"
    }
type result'_switch t u =
    {
        Gleam : $'Result(`t, `u)'
        Lua : result t u
        Fsharp : $'Result<`t, `u>'
        Python : result t u
        Rust : $'Result<`t, `u>'
        TypeScript : result t u
        Zig : result t u
    }
nominal result' t u = $'backend_switch `(result'_switch t u)'

inl result'_union () : bool =
    backend_switch {
        Gleam = fun () => false
        Lua = fun () => true
        Fsharp = fun () => false
        Python = fun () => true
        TypeScript = fun () => true
        Cpp = fun () => false
        CppHost = fun () => false
        Delphi = fun () => false
        Cuda = fun () => false
        Rust = fun () => false
        Zig = fun () => true
    }

inl result'_python () : bool =
    backend_switch {
        Gleam = fun () => false
        Lua = fun () => false
        Fsharp = fun () => false
        Python = fun () => true
        TypeScript = fun () => false
        Cpp = fun () => false
        CppHost = fun () => false
        Delphi = fun () => false
        Cuda = fun () => false
        Rust = fun () => false
        Zig = fun () => false
    }

inl result'_zig () : bool =
    backend_switch {
        Gleam = fun () => false
        Lua = fun () => false
        Fsharp = fun () => false
        Python = fun () => false
        TypeScript = fun () => false
        Cpp = fun () => false
        CppHost = fun () => false
        Delphi = fun () => false
        Cuda = fun () => false
        Rust = fun () => false
        Zig = fun () => true
        Lean = fun () => false
        C = fun () => false
        Wasm = fun () => false
        Bend = fun () => false
    }

inl result'_error_text forall e. (prefix : string) (e : e) : string =
    $'(std.fmt.allocPrint(spiral_gpa, "{s}{any}", .{ !prefix, !e }) catch \@panic("out of memory"))'

### unbox

In [ ]:
inl unbox forall t u. (x : result' t u) : result t u =
    if result'_union ()
    then $'!x ' : result t u
    else
        inl ok x : result t u = Ok x
        inl error x : result t u = Error x
        inl ok = join ok
        inl error = join error
        backend_switch {
            Gleam = fun () =>
                $'case !x { Ok(x) -> !ok(x) Error(x) -> !error(x) }' : result t u
            Lua = fun () =>
                $'local resultm_ok = !ok '
                $'local resultm_error = !error '
                $'local r = nil'
                $'if !x ~= nil then'
                $'    r = resultm_ok(!x)'
                $'else'
                $'    r = resultm_error(!x)'
                $'end'
                $'r' : result t u
            Fsharp = fun () =>
                real
                    typecase t with
                    | () => $'match !x with Ok () -> !ok () | Error x -> !error x' : result t u
                    | _ => $'match !x with Ok x -> !ok x | Error x -> !error x' : result t u
                : result t u
            Python = fun () =>
                $'ok = !ok '
                $'error = !error '
                $'r = None'
                $'if !x is not None:'
                $'    r = ok(!x)'
                $'else:'
                $'    r = error(!x)'
                $'r' : result t u
            Rust = fun () =>
                real
                    typecase t with
                    | () => $'match !x { Ok(()) => !ok(), Err(e) => !error(e) }' : result t u
                    | _ => $'match !x { Ok(x) => !ok(x), Err(e) => !error(e) }' : result t u
                : result t u
        }

### box

In [ ]:
inl box forall t u. (x : result t u) : result' t u =
    if result'_union ()
    then x |> dyn |> fun x => $'!x ' : result' t u
    else
        match x with
        | Ok x =>
            backend_switch {
                Gleam = fun () => $'Ok(!x)' : result' t u
                Lua = fun () => $'!x ' : result' t u
                Fsharp = fun () => $'Ok !x ' : result' t u
                Python = fun () => $'!x ' : result' t u
                Rust = fun () => $'Ok::<`t, `u>(!x)' : result' t u
            }
        | Error err =>
            backend_switch {
                Gleam = fun () => $'Error(!err)' : result' t u
                Lua = fun () => $'!err ' : result' t u
                Fsharp = fun () => $'Error !err ' : result' t u
                Python = fun () => $'!err ' : result' t u
                Rust = fun () => $'Err::<`t, `u>(!err)' : result' t u
            }

### map'''

In [ ]:
inl map''' forall t u e. (fn : t -> u) (x : result' t e) : result' u e =
    x |> unbox |> map fn |> box

In [1]:
///- --test
///> gleam
///> lua
///> fsharp
///> cuda
///> rust
///> typescript

inl x : _ _ string = Ok 3i32
inl y : _ i32 _ = Error "error1"
inl x' = x |> box |> unbox
inl y' = y |> box |> map''' id |> unbox
(x', y') |> _assert_eq' (x, y)

.gleam (gleam)
__assert_eq' { expected = (Ok(3), Error(error1)) }
.lua (lua)
__assert_eq' { expected = (Ok(3), Error(error1)) }
.fsx (fsharp)
__assert_eq' { expected = (Ok(3), Error(error1)) }
.py (cuda)
__assert_eq' { expected = (Ok(3), Error(error1)) }
.rs (rust)
__assert_eq' { expected = (Ok(3), Error(error1)) }
.ts (typescript)
__assert_eq' { expected = (Ok(3), Error(error1)) }

0

## rust

### anyhow_result

In [ ]:
nominal anyhow_result t =
    `(
        rust_type
            `($'anyhow_Result<`t>')
            `($'anyhow::Result<`t>')
            "type anyhow_Result<'T> = class end"
    )

### anyhow_error

In [ ]:
nominal anyhow_error =
    `(
        rust_type
            `($'anyhow_Error')
            `($'anyhow::Error')
            "type anyhow_Error = class end"
    )

inl anyhow_error error : anyhow_error =
    backend_switch {
        Rust = fun () => !\\(error, $'"anyhow::anyhow\!(\\\"{}\\\", $0)"') : anyhow_error
    }

### try'

In [ ]:
inl try' forall t u. (x : result' t u) : t =
    inl is_unit =
        real
            typecase t with
            | () => true
            | _ => false
    if result'_union ()
    then
        match unbox x with
        | Ok x => x
        | Error e =>
            inl msg : string =
                if result'_zig ()
                then result'_error_text "resultm.try' / e: " e
                elif result'_python ()
                then $'f"resultm.try\' / e: {!e}"'
                else $'"resultm.try\' / e: " .. tostring(!e)'
            failwith msg
    elif is_unit
    then (!\\(x, $'"true; $0?"') : bool) |> fun _ => $''
    else !\\(x, $'"$0?"')

### to_try

In [ ]:
inl to_try forall t u. (x : result' t u) : rust.try t =
    !\\(x, $'"$0"')

### unwrap'

In [ ]:
inl unwrap' forall t u. (x : result' t u) : t =
    if result'_union ()
    then
        match unbox x with
        | Ok x => x
        | Error e =>
            inl msg : string =
                if result'_zig ()
                then result'_error_text "resultm.unwrap' / e: " e
                elif result'_python ()
                then $'f"resultm.unwrap\' / e: {!e}"'
                else $'"resultm.unwrap\' / e: " .. tostring(!e)'
            failwith msg
    else !\\(x, $'"$0.unwrap()"')

### unwrap''

In [ ]:
inl unwrap'' forall t u. (x : result' t u) : t =
    if result'_union ()
    then
        match unbox x with
        | Ok x => x
        | Error e =>
            inl msg : string =
                if result'_zig ()
                then result'_error_text "resultm.unwrap'' / e: " e
                elif result'_python ()
                then $'f"resultm.unwrap\'\' / e: {!e}"'
                else $'"resultm.unwrap\'\' / e: " .. tostring(!e)'
            failwith msg
    else
        run_target function
            | Rust => fun () => x |> unwrap'
            | _ => fun () => $'match !x with Ok x -> x | Error e -> failwith $"resultm.unwrap\'\' / e: {e}"'

### unwrap_err'

In [ ]:
inl unwrap_err' forall t u. (x : result' t u) : u =
    if result'_union ()
    then
        match unbox x with
        | Ok x =>
            inl msg : string =
                if result'_zig ()
                then result'_error_text "resultm.unwrap_err' / x: " x
                elif result'_python ()
                then $'f"resultm.unwrap_err\' / x: {!x}"'
                else $'"resultm.unwrap_err\' / x: " .. tostring(!x)'
            failwith msg
        | Error x => x
    else $'match !x with Ok x -> failwith $"resultm.unwrap_err\' / x: %A{x}" | Error x -> x'

### unbox'

In [ ]:
inl unbox' forall t u. (x : result' t u) : result t u =
    unbox x

### map'

In [ ]:
inl map' forall t e u. (fn : t -> u) (source : result' t e) : result' u e =
    if result'_union ()
    then source |> unbox |> map fn |> box
    else
        (!\\(source, $'"true; let _result_map_ = $0.map(|x| { //"') : bool) |> ignore
        (!\\(fn !\($'"x"'), $'"true; $0 })"') : bool) |> ignore
        !\($'"_result_map_"')

### map''

In [ ]:
inl map'' forall t e u. (fn : t -> u) (source : result' t e) : result' u e =
    if result'_union ()
    then source |> unbox |> map fn |> box
    else
        inl fn = join fn
        inl source = join source
        !\($'"!source.map(|x| !fn(x))"')

### map_error'

In [ ]:
inl map_error' forall t e u. (fn : e -> u) (source : result' t e) : result' t u =
    if is_rust ()
    then
        inl fn = join fn
        !\\((source, fn), $'"$0.map_err(|x| $1(x))"')
    else source |> unbox |> map_error fn |> box

### map_error''

In [ ]:
inl map_error'' forall t e u. (fn : e -> u) (source : result' t e) : result' t u =
    if result'_union ()
    then source |> unbox |> map_error fn |> box
    else
        (!\\(source, $'"true; let _result_map_error__ = $0.map_err(|x| { //"') : bool) |> ignore
        (!\\(fn !\($'"x"'), $'"true; $0 })"') : bool) |> ignore
        !\($'"_result_map_error__"')

### map_error'''

In [ ]:
inl map_error''' forall t e u. (fn : e -> u) (source : result' t e) : result' t u =
    if result'_union ()
    then source |> unbox |> map_error fn |> box
    else !\\((source, fn), $'"$0.map_err(|x| $1(x))"')

### option_ok_or

In [ ]:
inl option_ok_or forall t e. (e : e) (source : optionm'.option' t) : result' t e =
    if result'_union ()
    then source |> optionm'.unbox |> from_option_error e |> box
    else !\\(source, $'"$0.ok_or(!e)"')

### unwrap_or_else

In [ ]:
inl unwrap_or_else forall t e u. (fn : e -> u) (source : result' t e) : u =
    (!\\(source, $'"true; let _result_unwrap_or_else = $0.unwrap_or_else(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'"true; $0 })"') : bool) |> ignore
    !\($'"_result_unwrap_or_else"')

### map_or_else

In [ ]:
inl map_or_else forall t e u v. (fn : e -> v) (fn2 : u -> v) (source : result' t e) : v =
    (!\\(source, $'"true; let _result_map_or_else = $0.map_or_else(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'"true; $0 }, |x| { //"') : bool) |> ignore
    (!\\(fn2 !\($'"x"'), $'"true; $0 })"') : bool) |> ignore
    !\($'"_result_map_or_else"')

### as_ref

In [ ]:
inl as_ref forall t e. (source : result' t e) : result' (rust.ref t) (rust.ref e) =
    !\($'"!source.as_ref()"')

### as_ref'

In [ ]:
inl as_ref' forall t e. (source : rust.ref (result' t e)) : result' (rust.ref t) (rust.ref e) =
    !\($'"!source.as_ref()"')

### unwrap_or'

In [ ]:
inl unwrap_or' forall t u. (default : t) (x : result' t u) : t =
    if result'_union ()
    then
        match unbox x with
        | Ok x => x
        | Error _ => default
    else !\\((x, default), $'"$0.unwrap_or($1)"')

### expect

In [ ]:
inl expect forall t u. (error : rust.ref string) (x : result' t u) : t =
    !\($'"!x.expect(&!error)"')

### is_err

In [ ]:
inl is_err forall t u. (x : result' t u) : bool =
    if is_rust ()
    then !\\(x, $'"$0.is_err()"')
    else
        match unbox x with
        | Ok _ => false
        | Error _ => true

### ok'

In [ ]:
inl ok' forall t. (x : result' t _) : optionm'.option' t =
    if is_rust ()
    then !\\(x, $'"$0.ok()"')
    else x |> unbox |> ok |> optionm'.box

### err

In [ ]:
inl err forall t u. (x : u) : result' t u =
    if is_rust ()
    then !\\(x, $'"Err($0)"')
    else (Error x : result t u) |> box

### ok''

In [ ]:
inl ok'' forall t u. (x : t) : result' t u =
    if is_rust ()
    then !\\(x, $'"Ok($0)"')
    else (Ok x : result t u) |> box

### transpose

In [ ]:
inl transpose forall t u. (x : optionm'.option' (result' t u)) : result' (optionm'.option' t) u =
    if result'_union ()
    then
        inl r : result (optionm'.option' t) u =
            match optionm'.unbox x with
            | Some x =>
                match unbox x with
                | Ok x => Ok (optionm'.some' x)
                | Error e => Error e
            | None => Ok (optionm'.none' ())
        box r
    else !\\(x, $'"$0.transpose()"')

### rc_try_unwrap

In [2]:
inl rc_try_unwrap forall t. (x : rust.rc t) : result' t (rust.rc t) =
    !\\(x, $'"std::rc::Rc::try_unwrap($0)"')

In [3]:
///- --test
///> rust

rust.new_rc true
|> rc_try_unwrap
|> unbox
|> _assert_eq (Ok true)

__assert_eq { expected = Ok(true) }

0